# Week 3a.1: Short-term Memory

Our agents so far have amnesia. Every `invoke` starts from nothing: the model does not remember what you asked in the previous message. 

Today we will address this problem for a single conversation.
- This doesn't apply to memory across conversations and sessions

In [1]:
from dotenv import load_dotenv
import os
import logging

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."

# silence a noisy advisory warning from the Google SDK
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

print("API key loaded")

API key loaded


## 0. Model & Tool Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [5]:
import requests
from langchain.tools import tool
from langchain_tavily import TavilySearch
from typing import Dict, Any
from datetime import datetime

# WEB SEARCH
@tool
def search_the_web(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    tavily = TavilySearch(max_results=3)
    query_dict = {"query": query}
    results = tavily.invoke(query_dict)
    return results

# CURRENT TIME
@tool
def get_current_time() -> str:
    """Return the current local date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

# WEATHER
@tool
def get_weather(city: str) -> str:
    """Get the weather data for the city provided as an argument"""

    data = requests.get(f"https://wttr.in/{city}?format=j1").json()
    return data["current_condition"][0]

## 1. The problem

In [6]:
response = model.invoke("Hello from Boston.")
print(response.text)

Hello Boston! How are things in Beantown today?


In [7]:
response = model.invoke("Where am I located?")
print(response.text)

I don't know your exact location because I don't have access to GPS or IP geolocation data for privacy and security reasons. 

However, if you're looking for your approximate location, you can check your device's settings or type "What is my location?" into a search engine.


The model has no idea. It is **stateless**: nothing carries over from one call to the next. What looked like a conversation in a chat app was never memory inside the model; the application was resending the history every time.

We already have the tool for this: a call can take a **list of messages**. That list is the memory.

That is all short-term memory is: the application rereads the entire conversation to the model on every single call. Nothing is stored inside the model.

## 2. Agents with Threads

For agents, LangChain provides a built-in method to save previous messages into the agent's state (or memory).
- https://docs.langchain.com/oss/python/langchain/short-term-memory

We need to add a **checkpointer** (InMemorySaver) when invoking the `create_agent` method, which will save the conversation state after every call, filed under a `thread_id` we choose. The `thread_id` is typically the unique conversation ID.

Messages with the same thread are saved to the same memory

Add this to agent **declaration**: 
- `checkpointer = InMemorySaver()`

The thread_id is passed as a configurable to agent **invocation**:
- `config = {"configurable": {"thread_id": "1"}}`


In [8]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[get_weather, search_the_web, get_current_time],
    system_prompt="You're a helpful assistant who answers users' questions concisely.",
    #TODO: add the checkpointer
    checkpointer = InMemorySaver()
)


In [9]:
#TODO define the config
config = {"configurable": {"thread_id": "1"}}

# then add this after the messages dictionary in your agent invocation

In [11]:
from langchain.messages import HumanMessage

message = [HumanMessage(content="Hello from Boston!")]

result = agent.invoke({"messages": message},config)
print(result["messages"][-1].text)

Hello from Boston! How can I help you today?


In [12]:
message2 = [HumanMessage(content="Where am I located?")]
result = agent.invoke({"messages": message2}, config)
print(result["messages"][-1].text)

Based on your greeting, you are in **Boston**, Massachusetts!


In [13]:
result

{'messages': [HumanMessage(content='Hello from Boston!', additional_kwargs={}, response_metadata={}, id='04c3938b-d36c-4ec4-a82d-a7b180a71111'),
  AIMessage(content=[{'type': 'text', 'text': 'Hello from Boston! How can I help you today?', 'extras': {'signature': 'EmAKXgFpFH0T0D9BbnbtsrIsRNyBsogVhIrmfLMsFyUMv3/EqH5iBpYRJkhjpBlaIReEZ7X3X7wcP9/5MkCPEmSJCbXHkYGUwWUAQDy7qZL4bKzrniqjW7XLk5/YtWbEy74='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d96f-faba-7100-be78-06d98d42e576-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 133, 'output_tokens': 11, 'total_tokens': 144, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='Where am I located?', additional_kwargs={}, response_metadata={}, id='9231a0fe-a590-4cec-8c09-e8d39ae4d97b'),
  AIMessage(content=[{'type': 'text', 'text': 'Based on your greeting, you are in **Bost

In [14]:
message3 = [HumanMessage(content="What's the weather like?")]

result = agent.invoke({"messages": message3}, config)
print(result["messages"][-1].text)

It's currently 60°F (15°C) and overcast in Boston, with winds coming from the NNE at around 18 mph.


In [15]:
result

{'messages': [HumanMessage(content='Hello from Boston!', additional_kwargs={}, response_metadata={}, id='04c3938b-d36c-4ec4-a82d-a7b180a71111'),
  AIMessage(content=[{'type': 'text', 'text': 'Hello from Boston! How can I help you today?', 'extras': {'signature': 'EmAKXgFpFH0T0D9BbnbtsrIsRNyBsogVhIrmfLMsFyUMv3/EqH5iBpYRJkhjpBlaIReEZ7X3X7wcP9/5MkCPEmSJCbXHkYGUwWUAQDy7qZL4bKzrniqjW7XLk5/YtWbEy74='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d96f-faba-7100-be78-06d98d42e576-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 133, 'output_tokens': 11, 'total_tokens': 144, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='Where am I located?', additional_kwargs={}, response_metadata={}, id='9231a0fe-a590-4cec-8c09-e8d39ae4d97b'),
  AIMessage(content=[{'type': 'text', 'text': 'Based on your greeting, you are in **Bost

Let's now pass in a different thread:

In [16]:
config = {"configurable": {"thread_id": "2"}}
result = agent.invoke({"messages": message2}, config)
print(result["messages"][-1].text)

I don't have access to your GPS or location data, so I can't tell you where you are located. If you'd like, you can let me know where you are!


## 3. A chat loop

 Let's look at how the chat loop works. Type `quit` to stop.

In [17]:
config = {"configurable": {"thread_id": "live-chat"}}

while True:
    user = input("You: ")

    if user.lower() in {"quit", "exit"}:
        break

    # TODO: turn user input into a human message
    message = [HumanMessage(content=user)]

    # TODO: call the agent with the human message and config
    result = agent.invoke({"messages": message},config)
    
    print("Assistant:", result["messages"][-1].text)

Assistant: The weather in Boston is currently 60°F (15°C) and overcast, with winds blowing at around 18 mph.
Assistant: Tomorrow in Boston, you can expect windy and rainy conditions with temperatures reaching a high around 61°F (16°C) and a low of about 55°F (13°C).
Assistant: The weather in Istanbul is currently 67°F (20°C) and overcast, with a wind speed of about 14 mph.


## 4. A chat window

The terminal loop above works, but one import gives us a real chat interface. `agentui.py` lives next to this notebook: it wraps any agent built with `create_agent`, shows tool calls as collapsible entries while the agent works, and sends the same `thread_id` on every turn, so the agent's own checkpointer does the remembering.

Run the cell and open the local URL it prints. Interrupt the kernel (or restart it) to stop the server.

In [20]:
# Quick Fix
import sys
sys.path.insert(0, "/Users/Teoman/Desktop/Classes/NU/Fall26/Agentic/Agentic-AI/W03a")

from agentui import GradioUI

app = GradioUI(agent, {"configurable": {"thread_id": "gradio-demo"}}, title="Agent with Memory")
app.launch()

/Users/Teoman/Desktop/Classes/NU/Fall26/Agentic/Agentic-AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 5. Tracing

[LangSmith](https://smith.langchain.com/) is LangChain's official tool for tracing agent calls and executions:

To set it up, get an API key from https://smith.langchain.com/ and add it your .env file.
- Set `LANGSMITH_TRACING=true`